# CatBoost

### 1. Định nghĩa CatBoost
CatBoost là một thuật toán học máy thuộc loại cây tăng cường gradient (Gradient Boosting Decision Tree - GBDT). Nó được phát triển bởi Yandex. Tên gọi CatBoost là viết tắt của "Categorical Boosting". 

### 2. Nguyên tắc hoạt động
CatBoost giải quyết các vấn đề thường gặp trong các thuật toán Gradient Boosting Decision Tree (GBDT): 
* **Xử lý các biến phân loại (Categorical Features):** CatBoost sử dụng một cách tiếp cận không cần tiền xử lý (như mã hóa One-Hot) cho các biến phân loại. Nó thực hiện việc mã hóa các biến phân loại khi huấn luyện.
* **Giải quyết vấn đề dự đoán thiên lệch (Prediction Shift/Target Leakage):** Đây là vấn đề xảy ra khi các giá trị mục tiêu (là giá nhà, giá trị liên tục trong bài toán Hồi quy) được sử dụng để tính toán các thống kê cho các biến phân loại, dẫn đến sự sai lệch. CatBoost sử dụng một phương pháp biến đổi gọi là Ordered Target Statistics (thống kê mục tiêu có thứ tự) để tránh rò rỉ dữ liệu mục tiêu. Phương pháp này sử dụng tất cả các mẫu huấn luyện trước một mẫu cụ thể để tính toán thống kê cho biến phân loại đó.
* **Giải quyết vấn đề độ lệch do ước tính gradient (Prediction Shift):** CatBoost sử dụng một thuật toán gọi là Ordered Boosting để tạo ra một ước tính không thiên lệch cho gradient. Nó xây dựng mô hình trên một tập hợp con dữ liệu, sau đó sử dụng mô hình đó để ước tính gradient trên tập hợp dữ liệu gốc.

### 3. Ứng dụng CatBoost
* **Hồi quy (Regression):** Dự đoán giá trị liên tục (ví dụ: dự đoán giá nhà).
* **Phân loại nhị phân (Binary Classification):** Phân loại dữ liệu thành hai lớp.
* **Phân loại đa lớp (Multi-class Classification):** Phân loại dữ liệu thành nhiều lớp.
* **Xếp hạng (Ranking) / Học để xếp hạng (Learning to Rank):** Sắp xếp các đối tượng theo mức độ liên quan (ví dụ: kết quả tìm kiếm).

### 4. Lý do sử dụng CatBoost
* **Độ chính xác cao:** CatBoost thường đạt hiệu suất cạnh tranh hoặc tốt hơn so với các thuật toán GBDT khác như XGBoost và LightGBM.
* **Xử lý tự động biến phân loại:** CatBoost tự động xử lý các biến phân loại mà không cần mã hóa thủ công (như One-Hot Encoding), giúp đơn giản hóa quy trình tiền xử lý dữ liệu.
* **Tránh rò rỉ mục tiêu:** Sử dụng kỹ thuật Ordered Boosting và Ordered Target Statistics giúp giảm thiểu hiện tượng dự đoán thiên lệch và rò rỉ dữ liệu mục tiêu, dẫn đến mô hình tổng quát hóa tốt hơn.
* **Khả năng chịu lỗi (Robustness):** CatBoost ít cần điều chỉnh siêu tham số hơn so với các mô hình GBDT khác và có thể đạt được kết quả tốt với các tham số mặc định.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostRegressor
import time

---
### Bước 1 & 2 & 3: Tiền xử lý dữ liệu
Trong phần này chúng ta thực hiện:
1. Đọc file `Dataset.csv` và `Bertdataset.npy`.
2. Lọc các outliers về giá và diện tích.
3. Loại bỏ các bài đăng rác (cho thuê, bán đất trống).
4. Điền các giá trị thiếu và chuẩn hóa dữ liệu.

In [2]:
# ==========================================
# BƯỚC 1: ĐỌC DỮ LIỆU
# ==========================================
print("Đang tải dữ liệu...")
df = pd.read_csv("Dataset.csv", encoding='utf-8-sig')
embeddings = np.load("Bertdataset.npy") 

# ==========================================
# BƯỚC 2: LỌC OUTLIERS & LOẠI BỎ TIN CHO THUÊ / ĐẤT
# ==========================================
print(f"Số lượng dữ liệu ban đầu: {len(df)} dòng")

# 2.1 Lọc Giá (1 -> 60 tỷ) và Diện tích (0 -> 1000m2)
mask_price_area = (df['Price_Billion'] <= 60) & (df['Price_Billion'] >= 1) & (df['Area'] > 0) & (df['Area'] <= 1000)

# 2.2 Lọc từ khóa: Dấu ngã (~) ở trước nghĩa là GIỮ LẠI NHỮNG DÒNG KHÔNG CHỨA TỪ KHÓA
keywords = r'cho thuê|bán đất|đất nền|đất trống|nhà trọ|phòng trọ'
mask_text = ~df['Title'].str.contains(keywords, case=False, na=False)

# Gộp tất cả điều kiện
final_mask = mask_price_area & mask_text

# Áp dụng bộ lọc
df = df[final_mask].copy()
embeddings = embeddings[final_mask]
print(f"Số lượng dữ liệu sau khi lọc: {len(df)} dòng")

# ==========================================
# BƯỚC 3: XỬ LÝ KHUYẾT THIẾU VÀ CHUẨN HÓA
# ==========================================
tabular_features = ['Area', 'Bedrooms', 'Bathrooms', 'Location']
df[tabular_features] = df[tabular_features].fillna(0)

print("Chuẩn hóa...")
scaler = StandardScaler()
cols_to_scale = ['Area', 'Bedrooms', 'Bathrooms']
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

# Khai báo X và y
X_tabular = df[tabular_features].values
X_full = np.hstack((X_tabular, embeddings)) 
y = df['Price_Billion'].values

# Chia tập Train / Test
X_train_full, X_test_full, y_train, y_test = train_test_split(X_full, y, test_size=0.2, random_state=42)

Đang tải dữ liệu...
Số lượng dữ liệu ban đầu: 8461 dòng
Số lượng dữ liệu sau khi lọc: 7676 dòng
Chuẩn hóa...


---
### Bước 4: Huấn luyện mô hình CatBoost
Sau khi chuẩn bị dữ liệu, chúng ta thiết lập các tham số siêu việt (hyperparameters) và sử dụng thuật toán CatBoost để tìm kiếm sự ánh xạ giữa các đặc trưng đầu vào và giá nhà thực tế.

In [3]:
# ==========================================
# BƯỚC 4: HUẤN LUYỆN MÔ HÌNH CATBOOST
# ==========================================
print("Bắt đầu huấn luyện mô hình CatBoost...")
start_time = time.time()

model = CatBoostRegressor(
    iterations=1000, 
    learning_rate=0.05, 
    depth=6, 
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50
)

model.fit(
    X_train_full, y_train,
    eval_set=(X_test_full, y_test),
    use_best_model=True
)

training_time = time.time() - start_time
print(f"Thời gian huấn luyện: {training_time:.2f} giây")

Bắt đầu huấn luyện mô hình CatBoost...
0:      learn: 13.3120765       test: 13.0318349        best: 13.0318349 (0) total: 200ms    remaining: 3m 19s
100:    learn: 6.6891067        test: 7.0420064 best: 7.0420064 (100)total: 3.64s    remaining: 32.4s
200:    learn: 5.7558275        test: 6.7238189 best: 6.7238189 (200)total: 7.03s    remaining: 27.9s
300:    learn: 4.9519326        test: 6.5618397 best: 6.5618397 (300)total: 10.6s    remaining: 24.6s
400:    learn: 4.3185289        test: 6.4738015 best: 6.4738015 (400)total: 14s      remaining: 20.9s
500:    learn: 3.8140790        test: 6.4120290 best: 6.4120290 (500)total: 17.4s    remaining: 17.3s
600:    learn: 3.3872678        test: 6.3594456 best: 6.3582541 (598)total: 22.4s    remaining: 14.9s
700:    learn: 3.0305437        test: 6.3350114 best: 6.3346223 (697)total: 30.1s    remaining: 12.8s
800:    learn: 2.7187547        test: 6.3030313 best: 6.3023964 (799)total: 33.8s    remaining: 8.39s
900:    learn: 2.4553043        te

---
### Bước 5: Đánh giá Mô Hình
Kiểm tra mức độ dự đoán sai số của mô hình so với tập kiểm tra chưa được học (test data).

In [4]:
# ==========================================
# BƯỚC 5: ĐÁNH GIÁ MÔ HÌNH
# ==========================================
y_pred = model.predict(X_test_full)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\n--- Kết quả đánh giá CatBoost ---")
print(f"RMSE: {rmse:.4f} (Lệch khoảng {rmse:.2f} tỷ)")
print(f"MAE:  {mae:.4f} (Lệch khoảng {mae:.2f} tỷ)")
print(f"R2:   {r2:.4f} (Độ chính xác tương đối: {r2*100:.2f}%)")


--- Kết quả đánh giá CatBoost ---
RMSE: 6.2635 (Lệch khoảng 6.26 tỷ)
MAE:  4.0617 (Lệch khoảng 4.06 tỷ)
R2:   0.7786 (Độ chính xác tương đối: 77.86%)


### 6. Kết quả thực tế và Phân tích mô hình CatBoost

Sau khi quá trình huấn luyện kết thúc ở vòng lặp thứ 996 (nhờ cơ chế Early Stopping), mô hình đã đạt được các chỉ số cực kỳ khả quan trên tập kiểm tra như sau:

* **RMSE (Root Mean Squared Error): 6.2635.** Trung bình các dự đoán của mô hình bị lệch khoảng 6.26 tỷ so với giá thực tế. Nhờ đặc tính phạt nặng các dự đoán sai số lớn của RMSE, con số này cho thấy mô hình không bị dự đoán sai lệch quá xa ở các bất động sản đắt tiền.
* **MAE (Mean Absolute Error): 4.0617.** Sai số tuyệt đối trung bình là 4.06 tỷ. Điều này phản ánh thực tế nhất độ chênh lệch giữa giá dự đoán và giá thực tế. So với một số lượng lớn bất động sản có giá trị lên tới 60 tỷ, sai số trung bình chỉ ~4 tỷ là một mức hoàn toàn chấp nhận được.
* **R² (R-squared): 0.7786 (77.86%).** Mức độ giải thích sự biến thiên của giá nhà đạt 77.86%. Nghĩa là mô hình đã nắm bắt và dự đoán đúng được gần 78% yếu tố ảnh hưởng đến giá trị của bất động sản.

**Đánh giá sự kết hợp giữa Đặc trưng bảng và BERT Embeddings:**
Việc sử dụng `np.hstack((X_tabular, embeddings))` giúp mô hình học được cả các thông số vật lý của bất động sản (Diện tích, số phòng, vị trí) cùng với ngữ nghĩa sâu xa từ tiêu đề tin đăng (nhờ thuật toán BERT). Với 7676 mẫu dữ liệu sau khi lọc, kết quả R² ~ 78% là một thành công đáng kể. Thuật toán CatBoost đã chứng tỏ được sức mạnh trong việc xử lý cực tốt vector nhúng văn bản có số chiều lớn, và tránh được hiện tượng quá khớp (overfitting) nhờ được kết hợp với siêu tham số `early_stopping_rounds`.